# Informe de Pruebas de Hipotesis Pareadas (Sigmoid vs Match Count)

Este documento compara estadisticamente los resultados disponibles en la carpeta results.

Objetivo: evaluar si sigmoid supera a match_count en exito del pipeline y tiempo de ejecucion.

## Diseno experimental y metodologia

Este analisis compara dos funciones de perdida del pipeline de regresion simbolica:
- `sigmoid`
- `match_count`

Cada caso de benchmark aparece evaluado en ambos metodos.

### Pares comparados
- sigmoid vs match_count

### Metricas evaluadas por par
- `success` (tasa de exito del pipeline).
- `time` (tiempo de ejecucion en segundos).

### Hipotesis por metrica (prueba t pareada, una cola)
- Para `success`: H1 = metodo A > metodo B.
- Para `time`: H1 = metodo A < metodo B (A es mas rapido).

Nivel de significancia: `alpha = 0.05`.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from scipy import stats

ALPHA = 0.05
METHODS = ["sigmoid", "match_count"]
PAIRS = [("sigmoid", "match_count")]
METRICS = ["success", "time"]

## 1) Deteccion de rutas y carga de resultados

Se buscan archivos results_*.json y se valida que los casos coincidan entre metodos.

In [2]:
def detect_results_base() -> Path:
    candidates = [
        Path('.'),
        Path('src/benchmark/results'),
        Path('../src/benchmark/results'),
        Path('../results'),
    ]
    required = [f'results_{method}.json' for method in METHODS]
    for cand in candidates:
        if all((cand / name).exists() for name in required):
            return cand
    raise FileNotFoundError(
        'No se encontraron results_sigmoid.json y results_match_count.json en las rutas conocidas.'
    )

base = detect_results_base()
files = {method: base / f'results_{method}.json' for method in METHODS}
print(f'Base de resultados: {base.resolve()}')
print('Archivos:', [f.name for f in files.values()])

data = {k: json.loads(v.read_text(encoding='utf-8')) for k, v in files.items()}
methods = list(files.keys())
maps = {m: {row['case_id']: row for row in rows} for m, rows in data.items()}

base_method = methods[0]
case_ids = sorted(maps[base_method].keys())
for method in methods[1:]:
    missing = sorted(set(case_ids) - set(maps[method].keys()))
    extra = sorted(set(maps[method].keys()) - set(case_ids))
    if missing or extra:
        raise ValueError(
            f'Casos inconsistentes en {method}. Faltan: {missing[:5]}, Extras: {extra[:5]}'
        )

common_cases = case_ids
print('Metodos detectados:', methods)
print(f'Casos usados en el analisis: {len(common_cases)}')

Base de resultados: /home/noel/Disco D/4to_Anno/tesis/Tesis/src/benchmark/results
Archivos: ['results_sigmoid.json', 'results_match_count.json']
Metodos detectados: ['sigmoid', 'match_count']
Casos usados en el analisis: 30


## 2) Preparacion del dataset pareado

Se arma una tabla con success_rate y tiempo promedio por caso, para cada metodo.

In [3]:
rows = []
for case_id in common_cases:
    row = {
        'case_id': case_id,
        'case_name': maps[base_method][case_id].get('case_name', str(case_id)),
    }
    for method in methods:
        entry = maps[method][case_id]
        row[f'{method}_success'] = float(entry.get('success_rate', np.nan))
        time_stats = entry.get('statistics', {}).get('time_seconds', {})
        row[f'{method}_time'] = float(time_stats.get('mean', np.nan))
    rows.append(row)

df = pd.DataFrame(rows)
df.head()

,case_id,case_name,sigmoid_success,sigmoid_time,match_count_success,match_count_time
0,1,system_linear_01,1.0,194.193555,1.0,446.437
1,2,system_linear_02,1.0,203.019239,1.0,470.000
2,3,system_linear_03,1.0,278.198586,1.0,1532.786
3,4,system_linear_04,1.0,190.234703,1.0,1630.541
4,5,system_linear_05,1.0,277.987487,1.0,501.570


## 3) Pruebas de hipotesis (una por bloque)

A continuacion se ejecutan 2 pruebas t pareadas (una cola), una por cada metrica.

En cada bloque se reporta explicitamente:
- hipotesis nula (H0)
- hipotesis alternativa (H1)
- estadisticos de prueba
- decision final (rechazar o no rechazar H0).

### Test 1: Sigmoid vs Match Count en exito del pipeline

Objetivo: evaluar si `sigmoid` tiene mayor tasa de exito que `match_count`.

- H0: \(\mu(\text{sigmoid} - \text{match_count}) \le 0\)
- H1: \(\mu(\text{sigmoid} - \text{match_count}) > 0\)

### Test 2: Sigmoid vs Match Count en tiempo de ejecucion

Objetivo: evaluar si `sigmoid` es mas rapido que `match_count` (menor tiempo promedio).

- H0: \(\mu(\text{sigmoid} - \text{match_count}) \ge 0\)
- H1: \(\mu(\text{sigmoid} - \text{match_count}) < 0\)

In [4]:
def run_single_hypothesis_test(df, method_a, method_b, metric, alpha=0.05):
    col_a = f'{method_a}_{metric}'
    col_b = f'{method_b}_{metric}'

    a = df[col_a].to_numpy(dtype=float)
    b = df[col_b].to_numpy(dtype=float)
    diff = a - b

    valid = ~np.isnan(diff)
    a = a[valid]
    b = b[valid]
    diff = diff[valid]

    if metric == 'success':
        alternative = 'greater'
        h0 = f'mu({method_a}-{method_b}) <= 0'
        h1 = f'mu({method_a}-{method_b}) > 0'
    elif metric == 'time':
        alternative = 'less'
        h0 = f'mu({method_a}-{method_b}) >= 0'
        h1 = f'mu({method_a}-{method_b}) < 0'
    else:
        raise ValueError(f'Metrica no soportada: {metric}')

    if len(diff) < 2:
        print('No hay suficientes datos para ejecutar la prueba.')
        return {
            'pair': f'{method_a} vs {method_b}',
            'metric': metric,
            'n': len(diff),
            't_stat': np.nan,
            'p_value': np.nan,
            'reject_h0': False,
            'mean_a': float(np.nanmean(a)) if len(a) else np.nan,
            'mean_b': float(np.nanmean(b)) if len(b) else np.nan,
            'mean_diff': float(np.nanmean(diff)) if len(diff) else np.nan,
            'h0': h0,
            'h1': h1,
        }

    if np.isclose(np.std(diff, ddof=1), 0.0):
        mean_diff = float(np.mean(diff))
        if alternative == 'greater':
            p_value = 0.0 if mean_diff > 0 else 1.0
        else:
            p_value = 0.0 if mean_diff < 0 else 1.0
        t_stat = np.inf if p_value == 0.0 else 0.0
    else:
        res = stats.ttest_rel(a, b, alternative=alternative, nan_policy='omit')
        t_stat = float(res.statistic)
        p_value = float(res.pvalue)
        mean_diff = float(np.mean(diff))

    reject_h0 = p_value < alpha

    print(f'Comparacion: {method_a} vs {method_b} | Metrica: {metric}')
    print(f'H0: {h0}')
    print(f'H1: {h1}')
    print(f'n pares: {len(diff)}')
    print(f'media({method_a}) = {np.mean(a):.6f}')
    print(f'media({method_b}) = {np.mean(b):.6f}')
    print(f'media({method_a} - {method_b}) = {mean_diff:.6f}')
    print(f't = {t_stat:.6f}')
    print(f'p-value = {p_value:.6g}')
    print(f"Decision (alpha={alpha}): {'RECHAZAR H0' if reject_h0 else 'NO rechazar H0'}")

    return {
        'pair': f'{method_a} vs {method_b}',
        'metric': metric,
        'n': len(diff),
        't_stat': t_stat,
        'p_value': p_value,
        'reject_h0': reject_h0,
        'mean_a': float(np.mean(a)),
        'mean_b': float(np.mean(b)),
        'mean_diff': mean_diff,
        'h0': h0,
        'h1': h1,
    }

In [5]:
test_1 = run_single_hypothesis_test(
    df=df,
    method_a='sigmoid',
    method_b='match_count',
    metric='success',
    alpha=ALPHA,
)

Comparacion: sigmoid vs match_count | Metrica: success
H0: mu(sigmoid-match_count) <= 0
H1: mu(sigmoid-match_count) > 0
n pares: 30
media(sigmoid) = 0.900000
media(match_count) = 0.700000
media(sigmoid - match_count) = 0.200000
t = 1.988604
p-value = 0.0281275
Decision (alpha=0.05): RECHAZAR H0


In [6]:
test_2 = run_single_hypothesis_test(
    df=df,
    method_a='sigmoid',
    method_b='match_count',
    metric='time',
    alpha=ALPHA,
)

Comparacion: sigmoid vs match_count | Metrica: time
H0: mu(sigmoid-match_count) >= 0
H1: mu(sigmoid-match_count) < 0
n pares: 30
media(sigmoid) = 414.045016
media(match_count) = 632.893667
media(sigmoid - match_count) = -218.848650
t = -2.099067
p-value = 0.0223152
Decision (alpha=0.05): RECHAZAR H0
